In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
df_train=pd.read_csv("./train_data.csv")
df_train=df_train[['group', 'ALB', 'GGT', 'TBIL', 'UCR']]
df_train
df_test=pd.read_csv("./test_data.csv")
df_test=df_test[['group', 'ALB', 'GGT', 'TBIL', 'UCR']]
df_test

,group,ALB,GGT,TBIL,UCR
0,0,44.5,18,18.0,5.30
1,0,41.4,13,10.8,8.16
2,0,42.4,16,16.4,8.38
3,0,44.3,30,17.2,9.75
4,0,46.2,15,16.7,12.26
...,...,...,...,...,...
376,1,43.3,20,13.8,9.07
377,1,38.3,18,10.9,8.97
378,1,39.4,11,14.2,9.66
379,1,44.5,14,12.8,8.48


In [11]:
X_train = df_train.drop(['group'], axis=1)  
y_train = df_train['group']  
X_test = df_test.drop(['group'], axis=1)
y_test =df_test['group']  


In [13]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression()
clf.fit(X_train, y_train)


LogisticRegression()

In [14]:
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, cohen_kappa_score, brier_score_loss, matthews_corrcoef
)

from sklearn.utils import resample
from scipy.stats import norm

def calculate_metrics_with_ci(y_true, y_pred, y_prob, n_bootstraps=1000, alpha=0.95):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    false_positive_rate = fp / (fp + tn) if (fp + tn) != 0 else 0
    false_negative_rate = fn / (fn + tp) if (fn + tp) != 0 else 0
    npv = tn / (tn + fn) if (tn + fn) != 0 else 0
    metrics = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "Specificity": specificity,
        "ROC-AUC": roc_auc_score(y_true, y_prob),
        "PR-AUC": average_precision_score(y_true, y_prob),
        "False Positive Rate": false_positive_rate,
        "False Negative Rate": false_negative_rate,
        "Kappa Coefficient": cohen_kappa_score(y_true, y_pred),
        "NPV": npv, 
        "Brier Score": brier_score_loss(y_true, y_prob),
        "MCC": matthews_corrcoef(y_true, y_pred)
    }
    # Bootstrap Calculate the confidence interval
    bootstrap_results = {metric: [] for metric in metrics}
    for _ in range(n_bootstraps):
        y_true_boot, y_pred_boot, y_prob_boot = resample(y_true, y_pred, y_prob)
        tn, fp, fn, tp = confusion_matrix(y_true_boot, y_pred_boot).ravel()
        specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
        false_positive_rate = fp / (fp + tn) if (fp + tn) != 0 else 0
        false_negative_rate = fn / (fn + tp) if (fn + tp) != 0 else 0
        npv_boot = tn / (tn + fn) if (tn + fn) != 0 else 0
               # Calculation metrics
        bootstrap_metrics = {
            "Accuracy": accuracy_score(y_true_boot, y_pred_boot),
            "Precision": precision_score(y_true_boot, y_pred_boot, zero_division=0),
            "Recall": recall_score(y_true_boot, y_pred_boot, zero_division=0),
            "F1 Score": f1_score(y_true_boot, y_pred_boot, zero_division=0),
            "Specificity": specificity,
            "ROC-AUC": roc_auc_score(y_true_boot, y_prob_boot),
            "PR-AUC": average_precision_score(y_true_boot, y_prob_boot),
            "False Positive Rate": false_positive_rate,
            "False Negative Rate": false_negative_rate,
            "Kappa Coefficient": cohen_kappa_score(y_true_boot, y_pred_boot),
            "NPV": npv_boot, 
            "Brier Score": brier_score_loss(y_true_boot, y_prob_boot),
            "MCC": matthews_corrcoef(y_true_boot, y_pred_boot)
        }
        for metric, value in bootstrap_metrics.items():
            bootstrap_results[metric].append(value)
    # Calculate the confidence interval
    ci_results = {}
    for metric, values in bootstrap_results.items():
        lower = np.percentile(values, (1 - alpha) / 2 * 100)
        upper = np.percentile(values, (1 + alpha) / 2 * 100)
        ci_results[metric] = f"{metrics[metric]:.3f} ({lower:.3f}, {upper:.3f})"
    
    return ci_results

datasets = {
    "Training Set": (X_train, y_train),
    "Testing Set": (X_test, y_test),
    }

# Calculate metrics for the training set, test set, and validation set
results_with_ci = {}

for name, (X, y) in datasets.items():
    y_prob = clf.predict_proba(X)[:, 1]  # Predict the probability of the positive class
    y_pred = clf.predict(X)             # Predictive label
    results_with_ci[name] = calculate_metrics_with_ci(y, y_pred, y_prob)
results_with_ci_df = pd.DataFrame(results_with_ci)
results_with_ci_df

,Training Set,Testing Set
Accuracy,"0.870 (0.847, 0.892)","0.848 (0.811, 0.885)"
Precision,"0.846 (0.802, 0.888)","0.807 (0.728, 0.882)"
Recall,"0.743 (0.694, 0.796)","0.719 (0.638, 0.793)"
F1 Score,"0.791 (0.755, 0.828)","0.760 (0.696, 0.814)"
Specificity,"0.933 (0.913, 0.951)","0.913 (0.876, 0.948)"
ROC-AUC,"0.930 (0.912, 0.947)","0.914 (0.885, 0.941)"
PR-AUC,"0.900 (0.874, 0.924)","0.866 (0.819, 0.906)"
False Positive Rate,"0.067 (0.049, 0.087)","0.087 (0.052, 0.124)"
False Negative Rate,"0.257 (0.204, 0.306)","0.281 (0.207, 0.362)"
Kappa Coefficient,"0.697 (0.647, 0.748)","0.649 (0.565, 0.727)"


In [15]:
results_with_ci_df.to_csv("./Model_evaluation_metrics.csv")